# ROGII Clean Artifact Well GBR Submission

This notebook builds the artifact-only clean submission system selected in exp100. It uses only the competition input and `ravaghi/wellbore-geology-prediction-artifacts`.

The artifact OOF model predictions are averaged as the base TVT prediction. The `target` column is used only for rows outside `sample_submission.csv` to learn a per-well residual offset; sample-submission rows are excluded from residual training and CV selection. A compact non-sample CV grid selects the well-level residual model and shrink weight, then the notebook writes `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
from collections.abc import Callable
import gc
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_NAMES = [
    "catboost-1",
    "catboost-2",
    "catboost-3",
    "lightgbm-1",
    "lightgbm-2",
    "lightgbm-3",
    "lightgbm-4",
]
WEIGHTS = np.linspace(-1.0, 1.0, 81)


def _candidate_paths():
    roots = []
    if Path("/kaggle/input").exists():
        roots.extend([
            Path("/kaggle/input/rogii-wellbore-geology-prediction"),
            Path("/kaggle/input/wellbore-geology-prediction-artifacts"),
            Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts"),
        ])
    roots.extend([
        Path("../data/raw/rogii-wellbore-geology-prediction"),
        Path("data/raw/rogii-wellbore-geology-prediction"),
        Path("../data/artifacts/wellbore-geology-prediction-artifacts"),
        Path("data/artifacts/wellbore-geology-prediction-artifacts"),
    ])
    return roots


def find_competition_root():
    for root in _candidate_paths():
        if (root / "sample_submission.csv").exists():
            return root
    if Path("/kaggle/input").exists():
        for sample_path in sorted(Path("/kaggle/input").glob("**/sample_submission.csv")):
            return sample_path.parent
    raise FileNotFoundError("Could not find competition sample_submission.csv")


def artifact_train_csv(root: Path) -> Path:
    nested = root / "data" / "train.csv"
    flat = root / "train.csv"
    if nested.exists():
        return nested
    if flat.exists():
        return flat
    return nested


def find_artifact_root():
    for root in _candidate_paths():
        if artifact_train_csv(root).exists():
            return root
    if Path("/kaggle/input").exists():
        for train_path in sorted(Path("/kaggle/input").glob("**/data/train.csv")):
            parent = train_path.parent.parent
            if any((parent / "models" / name / "oof_preds.pkl").exists() for name in MODEL_NAMES):
                return parent
        for train_path in sorted(Path("/kaggle/input").glob("**/train.csv")):
            parent = train_path.parent
            if any((parent / f"{name}_oof_preds.pkl").exists() for name in MODEL_NAMES):
                return parent
    raise FileNotFoundError("Could not find artifact dataset train.csv and OOF files")


def load_oof_delta(root: Path, model_name: str) -> np.ndarray:
    nested = root / "models" / model_name / "oof_preds.pkl"
    flat = root / f"{model_name}_oof_preds.pkl"
    if nested.exists():
        path = nested
    elif flat.exists():
        path = flat
    else:
        raise FileNotFoundError(f"Missing OOF artifact for {model_name}")
    return np.asarray(joblib.load(path), dtype=np.float32)


def rmse(pred: np.ndarray, y: np.ndarray) -> float:
    err = pred.astype(np.float64) - y.astype(np.float64)
    return float(np.sqrt(np.mean(err * err)))


def build_well_meta(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    core_tokens = ("last_known_tvt", "pf_", "beam_", "sc", "hyb", "sig_", "tw_", "gr", "frm_rmse")
    core_cols = [c for c in feature_cols if any(token in c for token in core_tokens)]
    agg = {c: ["mean"] for c in feature_cols}
    for col in core_cols:
        agg[col].append("std")
    meta = df[["well"] + feature_cols].groupby("well", sort=True).agg(agg)
    meta.columns = ["__".join(col).strip("_") for col in meta.columns.to_flat_index()]
    row_count = df.groupby("well", sort=True).size().rename("row_count").astype("float32")
    return meta.join(row_count).fillna(0.0)


def model_factories() -> dict[str, Callable[[], object]]:
    return {
        "ridge_1000": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1000.0, solver="lsqr", fit_intercept=True)),
        "rf_d4_l5": lambda: RandomForestRegressor(
            n_estimators=500, max_depth=4, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        "rf_d6_l5": lambda: RandomForestRegressor(
            n_estimators=500, max_depth=6, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        "rf_d8_l5": lambda: RandomForestRegressor(
            n_estimators=500, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        "et_d6_l5": lambda: ExtraTreesRegressor(
            n_estimators=500, max_depth=6, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        "et_d8_l5": lambda: ExtraTreesRegressor(
            n_estimators=500, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        "hgb_l2_1": lambda: HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.03, l2_regularization=1.0, max_leaf_nodes=15, random_state=42
        ),
        "gbr_d3": lambda: GradientBoostingRegressor(
            n_estimators=300, learning_rate=0.03, max_depth=3, min_samples_leaf=5, random_state=42
        ),
        "gbr_d4": lambda: GradientBoostingRegressor(
            n_estimators=300, learning_rate=0.03, max_depth=4, min_samples_leaf=5, random_state=42
        ),
    }


COMP_ROOT = find_competition_root()
ART_ROOT = find_artifact_root()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print(f"Competition root: {COMP_ROOT}")
print(f"Artifact root:    {ART_ROOT}")
print(f"Work root:        {WORK_ROOT}")

In [ ]:
sample = pd.read_csv(COMP_ROOT / "sample_submission.csv", usecols=["id"])
sample_ids = set(sample["id"].astype(str))
train_csv = artifact_train_csv(ART_ROOT)
columns = pd.read_csv(train_csv, nrows=0).columns.tolist()
required = {"well", "id", "last_known_tvt", "target"}
missing_required = sorted(required.difference(columns))
if missing_required:
    raise ValueError(f"Artifact train.csv is missing required columns: {missing_required}")

feature_cols = [c for c in columns if c not in ("well", "id", "target")]
dtypes = {c: "float32" for c in feature_cols + ["target"]}
dtypes.update({"well": "string", "id": "string"})
df = pd.read_csv(train_csv, dtype=dtypes)
if df["id"].duplicated().any():
    raise ValueError("Artifact train.csv has duplicate ids")

ids = df["id"].astype(str).to_numpy()
wells = df["well"].astype(str).to_numpy()
submit_mask = np.isin(ids, list(sample_ids))
if int(submit_mask.sum()) != len(sample):
    raise ValueError(f"Expected {len(sample)} sample ids in artifact train.csv, found {int(submit_mask.sum())}")

last_known = df["last_known_tvt"].to_numpy(np.float32)
y_abs = (last_known + df["target"].to_numpy(np.float32)).astype(np.float32)
pred_sum = np.zeros(len(df), dtype=np.float32)
loaded = []
for model_name in MODEL_NAMES:
    delta = load_oof_delta(ART_ROOT, model_name)
    if len(delta) != len(df):
        raise ValueError(f"{model_name} length mismatch: {len(delta)} != {len(df)}")
    pred_sum += last_known + delta
    loaded.append(model_name)
    print(f"Loaded {model_name}: {len(delta):,} rows")
base_pred = (pred_sum / np.float32(len(loaded))).astype(np.float32)

train_mask = ~submit_mask
residual = (y_abs - base_pred).astype(np.float32)
meta = build_well_meta(df, feature_cols)
well_ids = meta.index.astype(str).to_numpy()
Xw = meta.to_numpy(np.float32)
submit_wells = set(wells[submit_mask])
train_well_mask = ~np.isin(well_ids, list(submit_wells))
submit_well_mask = np.isin(well_ids, list(submit_wells))

well_target = (
    pd.DataFrame({"well": wells[train_mask], "residual": residual[train_mask]})
    .groupby("well", sort=True)["residual"]
    .mean()
    .reindex(well_ids)
)
train_well_positions = np.flatnonzero(train_well_mask & well_target.notna().to_numpy())
submit_well_positions = np.flatnonzero(submit_well_mask)
if len(submit_well_positions) == 0:
    raise ValueError("No sample-submission wells were found in artifact metadata")

well_target_values = well_target.fillna(0.0).to_numpy(np.float32)
well_to_pos = {well: pos for pos, well in enumerate(well_ids)}
row_well_pos = np.array([well_to_pos[well] for well in wells], dtype=np.int32)
folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(train_well_positions))

factories = model_factories()
results = []
for name, factory in factories.items():
    oof_well = np.zeros(train_well_positions.shape[0], dtype=np.float32)
    for tr_rel, va_rel in folds:
        tr_pos = train_well_positions[tr_rel]
        va_pos = train_well_positions[va_rel]
        model = factory()
        model.fit(Xw[tr_pos], well_target_values[tr_pos])
        oof_well[va_rel] = model.predict(Xw[va_pos]).astype(np.float32)
        del model
        gc.collect()

    correction_by_well = np.zeros(len(well_ids), dtype=np.float32)
    correction_by_well[train_well_positions] = oof_well
    row_correction = correction_by_well[row_well_pos]
    weight_scores = [
        {"weight": float(weight), "dev_cv_rmse": rmse(base_pred[train_mask] + weight * row_correction[train_mask], y_abs[train_mask])}
        for weight in WEIGHTS
    ]
    best_weight = min(weight_scores, key=lambda row: row["dev_cv_rmse"])
    row = {
        "name": name,
        "raw_dev_cv_rmse": rmse(base_pred[train_mask] + row_correction[train_mask], y_abs[train_mask]),
        "best_weight": best_weight["weight"],
        "best_dev_cv_rmse": best_weight["dev_cv_rmse"],
    }
    print(f"{name}: {row}")
    results.append(row)

selected = min(results, key=lambda row: row["best_dev_cv_rmse"])
final_model = factories[selected["name"]]()
final_model.fit(Xw[train_well_positions], well_target_values[train_well_positions])
submit_correction = final_model.predict(Xw[submit_well_positions]).astype(np.float32)
del final_model
gc.collect()

correction_by_well = np.zeros(len(well_ids), dtype=np.float32)
correction_by_well[submit_well_positions] = submit_correction
row_correction = correction_by_well[row_well_pos]
submit_pred = base_pred[submit_mask] + selected["best_weight"] * row_correction[submit_mask]

pred_frame = pd.DataFrame({"id": ids[submit_mask], "tvt": submit_pred.astype(np.float32)})
submission = sample.merge(pred_frame, on="id", how="left")
missing = int(submission["tvt"].isna().sum())
if missing:
    examples = submission.loc[submission["tvt"].isna(), "id"].head(10).tolist()
    raise ValueError(f"Missing predictions for {missing} sample ids; examples: {examples}")
if not np.isfinite(submission["tvt"].to_numpy(dtype=np.float64)).all():
    raise ValueError("Submission contains non-finite tvt values")

out_path = WORK_ROOT / "submission.csv"
submission.to_csv(out_path, index=False)
report = {
    "base_members": loaded,
    "selected": selected,
    "candidate_results": sorted(results, key=lambda row: row["best_dev_cv_rmse"]),
    "artifact_rows": int(len(df)),
    "train_rows_excluding_sample": int(train_mask.sum()),
    "submission_rows": int(len(submission)),
    "sample_wells": sorted(submit_wells),
    "ids_match_sample_order": bool(submission["id"].equals(sample["id"])),
    "missing_predictions": missing,
    "tvt_min": float(submission["tvt"].min()),
    "tvt_max": float(submission["tvt"].max()),
    "tvt_mean": float(submission["tvt"].mean()),
    "output": str(out_path),
}
(WORK_ROOT / "validation_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print(submission.head())